# Practice 7. Remaining useful life of a turbofan and the FMEA before and after

Industrial AI, KBTU. Goes with Lecture 11 and `inclass_work_11.ipynb`. **1.25 points.**

**The process.** NASA's C-MAPSS simulator ran a fleet of turbofan engines of one type. Each engine
starts with its own unknown wear, then a fault grows until the engine fails. **FD001** is the
simplest subset: one flight condition (sea level) and one failure mode, degradation of the
high-pressure compressor (HPC).

**The data** come from Kaggle (`behrad3d/nasa-cmaps`); the loader cell downloads them once.

| file | contents |
|---|---|
| `train_FD001.txt` | 100 engines, run to failure: the last row of each engine is its last working cycle |
| `test_FD001.txt` | 100 other engines, cut off some time before failure |
| `RUL_FD001.txt` | the true remaining life of each test engine at its last row |

Each row is one cycle (one flight): engine number, cycle number, 3 operating settings, 21
sensors. The 14 sensors that carry information:

| | | | |
|---|---|---|---|
| `s2` T24, LPC outlet temperature | `s3` T30, HPC outlet temperature | `s4` T50, LPT outlet temperature | `s7` P30, HPC outlet pressure |
| `s8` Nf, fan speed | `s9` Nc, core speed | `s11` Ps30, HPC static pressure | `s12` phi, fuel flow to Ps30 ratio |
| `s13` NRf, corrected fan speed | `s14` NRc, corrected core speed | `s15` BPR, bypass ratio | `s17` bleed enthalpy |
| `s20` W31, HPT coolant bleed | `s21` W32, LPT coolant bleed | | |

**The question.** Today the fleet is watched by a limit on one sensor. Would a model that predicts
the remaining useful life (RUL) catch failures better? How much engine life would it waste doing
so, and what does that do to the risk priority number in the FMEA?

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and sometimes
a variable with a result. How you write them is up to you, as long as you use the method the task
names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs, and the share of tests passed gives the points for the code. Code that only fits the
public cases loses those points.

Cells marked *Common ground* are written for you. Run them as they are: they fix the folds, the
features and the alarm window, so that everybody's numbers can be compared.

The methods are yours: RUL labels, the scoring function, the rolling slope, ridge regression by
the normal equations, gradient boosting with an asymmetric loss, the alarm logic and the Wilson
bound. `DecisionTreeRegressor` is allowed as the weak learner. Library versions of the rest are
for checking your result only.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_7.ipynb`;
2. `practice_7_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_7_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | total |
|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.08 | 0.08 | 0.12 | 0.15 | 0.06 | 0.08 | 0.10 | 0.67 |
| answers | 0.07 | 0.07 | 0.08 | 0.10 | 0.09 | 0.07 | 0.10 | 0.58 |

### The rules

- **Folds are by engine.** An engine is never on both sides of a split.
- **The test set is touched once**, in Task 5, after every choice has been made on the validation
  points. A decision changed after seeing the test score makes that score worthless.
- **No future values in a feature.** At cycle $t$ a feature may use cycles up to $t$ only.
- Scalers and limits are fitted on training engines only.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; the whole notebook runs in about three
  minutes on a laptop. Then check your answers against the new output.
- Figures need axis labels and a title that says what to look at.
- Stuck? `inclass_work_11.ipynb` goes through POD and FMEA on the hydraulic rig.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import sklearn
from sklearn.tree import DecisionTreeRegressor

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'cmapss': 'behrad3d/nasa-cmaps',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1


In [2]:
COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
DATA = next(dataset_folder("cmapss").rglob("train_FD001.txt")).parent


def read_cmapss(name):
    return pd.read_csv(DATA / f"{name}_FD001.txt", sep=r"\s+", header=None, names=COLS)


train = read_cmapss("train")
test = read_cmapss("test")
# the true RUL at the last row of each test engine
rul_test = np.loadtxt(DATA / "RUL_FD001.txt", dtype=int)
print("train", train.shape, "| test", test.shape,
      "| true RUL of the test engines", rul_test.shape)
train.head(3)

cmapss: Kaggle behrad3d/nasa-cmaps
train (20631, 26) | test (13096, 26) | true RUL of the test engines (100,)


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.001,-0.000,100.000,518.670,641.820,"1,589.700","1,400.600",14.620,21.610,554.360,"2,388.060","9,046.190",1.300,47.470,521.660,"2,388.020","8,138.620",8.419,0.030,392,2388,100.000,39.060,23.419
1,1,2,0.002,-0.000,100.000,518.670,642.150,"1,591.820","1,403.140",14.620,21.610,553.750,"2,388.040","9,044.070",1.300,47.490,522.280,"2,388.070","8,131.490",8.432,0.030,392,2388,100.000,39.000,23.424
2,1,3,-0.004,0.000,100.000,518.670,642.350,"1,587.990","1,404.200",14.620,21.610,554.260,"2,388.080","9,052.940",1.300,47.270,522.420,"2,388.030","8,133.230",8.418,0.030,390,2388,100.000,38.950,23.344


---

## Task 1. Run-to-failure data and the RUL label  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

In a run-to-failure record the last row of each engine has RUL = 0, so at cycle $t$ of an
engine that lived $T$ cycles

$$\text{RUL}(t) = T - t, \qquad \text{capped version: } \min\big(T - t,\ \text{cap}\big).$$

Plotted against the cycle number, the capped target is flat at the cap early in life and then
falls by one per cycle until failure: a piecewise-linear target (Heimes, 2008).

**Write**

- `add_rul(df, cap=None)`: a copy of `df` with a new integer column `"RUL"`, computed for each
  engine separately; with `cap`, limited from above by it. `df` itself stays unchanged, and the
  rows keep their order and index, whatever that order is.
- `lifetimes`: a `pd.Series` indexed by engine number, the number of cycles each training engine
  lived.
- `flat_columns`: a list of the columns among `op1`–`op3` and `s1`–`s21` of `train` that carry no
  information.

In [ ]:
def add_rul(df, cap=None):
    ...


lifetimes = ...
flat_columns = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_two_engines():
    demo = pd.DataFrame({"unit": [1, 1, 1, 2, 2], "cycle": [1, 2, 3, 1, 2]})
    rul = add_rul(demo)["RUL"].tolist()
    assert rul == [2, 1, 0, 1, 0], "RUL counts down to 0 at the last cycle of each engine"
    capped = add_rul(demo, cap=1)["RUL"].tolist()
    assert capped == [1, 1, 0, 1, 0], "cap limits the RUL from above"


test_two_engines()


def test_input_unchanged():
    demo = pd.DataFrame({"unit": [1, 1], "cycle": [1, 2]})
    got = add_rul(demo)
    ok = isinstance(got, pd.DataFrame) and "RUL" in got
    assert ok, "add_rul returns a DataFrame with a column RUL"
    unchanged = list(demo.columns) == ["unit", "cycle"]
    assert unchanged, "add_rul must return a copy, not change its argument"


test_input_unchanged()


def test_results_have_the_right_type():
    ok = isinstance(lifetimes, pd.Series) and len(lifetimes) == train["unit"].nunique()
    assert ok, "lifetimes: one value per training engine"
    ok = isinstance(flat_columns, list) and set(flat_columns) <= set(COLS[2:])
    assert ok, "flat_columns: a list of column names of train"


test_results_have_the_right_type()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
SENSORS = ["s2", "s3", "s4", "s7", "s8", "s9", "s11",
           "s12", "s13", "s14", "s15", "s17", "s20", "s21"]
CAP = 125                                     # the model learns min(RUL, 125)
WINDOW = 30                                   # cycles of history per feature (Task 3)

train = add_rul(train)                        # the true RUL stays in "RUL"
train["y"] = train["RUL"].clip(upper=CAP)     # the training target

# five folds by ENGINE, the same for everybody; an engine never appears on both sides
order = np.random.default_rng(7).permutation(train["unit"].unique())
fold_of = {u: i % 5 for i, u in enumerate(order)}
train["fold"] = train["unit"].map(fold_of)


def val_points(df, per_unit=5, seed=11):
    # Row labels of a test-like validation set: from each engine, cycles with a true RUL
    # of 7 to 145 (the range of the real test set) and a full window of history behind
    # them.
    rng = np.random.default_rng(seed)
    picks = []
    for _, g in df.groupby("unit", sort=True):
        g = g[g["RUL"].between(7, 145) & (g["cycle"] > WINDOW)]
        picks.extend(rng.choice(g.index.to_numpy(), per_unit, replace=False))
    return np.array(picks)


is_val = (train["fold"] == 0).to_numpy()      # fold 0 is the validation fold
idx_val = val_points(train[is_val])           # 20 engines x 5 = 100 validation points
print("engines per fold:", train.groupby("fold")["unit"].nunique().to_dict(),
      "| validation points:", len(idx_val))

**Questions** (answer them in `practice_7_answers.md`, Task 1, with the numbers your code printed):

1. How many engines are there, and how long do they live: minimum, median, maximum? Show a histogram of the lifetimes.
2. Which of the 21 sensors and 3 settings carry no information? Say how you decided.
3. Plot `s11` and `s4` against the true RUL for eight engines, RUL decreasing to the right. Where does the trend begin?
4. Why do we train on $\min(\text{RUL}, 125)$? Back it with the correlation of `s11` with the RUL on rows where RUL > 125 and where RUL ≤ 125.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 2. RMSE and the asymmetric score  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

Let $d_i = \hat{y}_i - y_i$, predicted minus true RUL. A positive $d$ means the model thinks the
engine will live longer than it will: **a late prediction**.

$$\text{RMSE} = \sqrt{\frac1n \sum_i d_i^2}, \qquad
S = \sum_i s_i, \quad s_i = \begin{cases} e^{-d_i/13} - 1, & d_i < 0, \\ e^{\,d_i/10} - 1, & d_i \ge 0. \end{cases}$$

The score is the PHM08 challenge metric (Saxena et al., 2008). Lower is better.

Expand the exponentials for small errors: $s \approx |d|/13$ on the early side and $s \approx
d/10$ on the late side. Near zero the score behaves like an absolute error with a steeper slope
on one side. For large errors the exponential takes over, so the ratio of the two sides keeps
growing with $|d|$.

**Write** `rmse(y_true, y_pred)` and `nasa_score(y_true, y_pred)`. Both take lists or arrays of the
same length and return a Python `float`.

In [ ]:
def rmse(y_true, y_pred):
    ...


def nasa_score(y_true, y_pred):
    ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_three_engines():
    y, p = [100, 100, 100], [110, 90, 100]
    assert abs(rmse(y, p) - 8.16497) < 1e-3, f"rmse: expected 8.165, got {rmse(y, p)}"
    score = nasa_score(y, p)
    assert abs(score - 2.8764) < 1e-3, f"nasa_score: expected 2.876, got {score}"


test_three_engines()


def test_perfect_prediction_scores_zero():
    assert rmse([5, 7], [5, 7]) == 0 and nasa_score([5, 7], [5, 7]) == 0


test_perfect_prediction_scores_zero()


def test_late_costs_more_than_early():
    late, early = nasa_score([50], [60]), nasa_score([50], [40])
    assert late > early, "a late prediction must cost more than an early one"
    ok = isinstance(nasa_score([1], [2]), float) and isinstance(rmse([1], [2]), float)
    assert ok, "return a float"


test_late_costs_more_than_early()


print("all tests passed")

**Questions** (answer them in `practice_7_answers.md`, Task 2, with the numbers your code printed):

1. Plot the score penalty of one engine against $d$ from −40 to +40, and $d^2$ scaled so that both agree at $d = -40$. Which side does the score punish more, and why would an airline want that?
2. Take the true RUL of the validation points, `train.loc[idx_val, "RUL"]`. Report RMSE and score of three predictions: everyone 15 cycles early, everyone 15 cycles late, half early and half late.
3. Take "everyone 15 cycles early" and make one engine 60 cycles late instead. What share of the total score comes from that one point, and what does it say about using this score alone?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Tasks 3 to 6: the rolling features. Run it as it is.*

In [10]:
def make_features(df, slope_fn, w=WINDOW):
    # One row per cycle: the rolling mean and the rolling slope of every sensor over the
    # last w cycles. Each engine separately; only past values are used.
    frames = []
    for _, g in df.groupby("unit", sort=False):
        data = {}
        for s in SENSORS:
            x = g[s].to_numpy(float)
            data[s + "_mean"] = pd.Series(x).rolling(w, min_periods=1).mean().to_numpy()
            data[s + "_slope"] = slope_fn(x, w)
        frames.append(pd.DataFrame(data, index=g.index))
    return pd.concat(frames).loc[df.index]

---

## Task 3. Features from the past and a linear baseline  
*0.20 points: 0.12 for the code (tests), 0.08 for the answers*

**Rolling slope.** The least-squares slope of the last $w$ values, $t = 0, \dots, w-1$:

$$b = \frac{\sum_t (t - \bar t)(x_t - \bar x)}{\sum_t (t - \bar t)^2}.$$

It is the covariance of the values with time divided by the variance of time. Noise that does
not trend with $t$ averages out in the numerator, so the slope of 30 noisy points is much more
stable than the difference of two of them. Until $w$ values exist the slope is set to 0: a line
through three noisy points says nothing.

**Ridge by the normal equations.** Stack the standardised features of the $n$ training rows into
$\mathbf{X} \in \mathbb{R}^{n \times p}$, the targets into $\mathbf{y}$, and let $\bar y$ be
their mean. Ridge minimises the squared error plus a penalty on the size of the weights:

$$\min_{\mathbf{w}}\ \lVert \mathbf{X}\mathbf{w} - (\mathbf{y} - \bar y) \rVert^2 + \lambda \lVert \mathbf{w} \rVert^2
\quad\Rightarrow\quad
(\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})\, \mathbf{w} = \mathbf{X}^\top (\mathbf{y} - \bar y), \qquad
\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + \bar y.$$

Set the gradient $2\mathbf{X}^\top(\mathbf{X}\mathbf{w} - \mathbf{y} + \bar y) + 2\lambda\mathbf{w}$
to zero and you get the system on the right. What does $\lambda$ do? Write
$\mathbf{X}^\top\mathbf{X} = \sum_j d_j \mathbf{v}_j \mathbf{v}_j^\top$ in its eigenvectors.
Along each direction $\mathbf{v}_j$ the ridge weight is the least-squares weight times
$d_j / (d_j + \lambda)$. Directions the data span well ($d_j \gg \lambda$) are left alone, and
directions where the data hardly vary, such as the difference of two nearly collinear sensors,
are shrunk towards zero. That is where plain least squares would put huge weights of opposite
sign. The intercept is not penalised, which is why it is handled separately as $\bar y$.

**Write**

- `rolling_slope(x, w)`: a 1-D array of the same length as `x`, the slope at each position, 0 at
  the first $w - 1$ positions;
- `fit_ridge(X, y, lam)`: returns `(w, y_mean)`, the weights from the normal equations (solve the
  system, do not invert the matrix) and the mean of `y`;
- `baseline_rmse`: a dict with the validation RMSE against the true RUL of three baselines, all
  trained on the other 80 engines, standardised with the mean and std of the training rows only,
  $\lambda = 1$, predictions clipped to $[0, 125]$:
  `"constant"` (the mean of `y_tr`), `"raw"` (ridge on the 14 sensors of the current cycle,
  `raw_tr` and `raw_val`) and `"rolling"` (ridge on `X_tr`, `X_val`).

`X_tr`, `y_tr`, `X_val`, `y_val`, `raw_tr`, `raw_val` come from the common cell under the tests,
which needs your `rolling_slope`. So write the two functions here, and `baseline_rmse` in the cell
for the questions, after the common cell has run.

In [ ]:
def rolling_slope(x, w):
    ...


def fit_ridge(X, y, lam):
    ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_slope_of_a_straight_line():
    s = rolling_slope(np.array([1., 3., 5., 7.]), 3)
    msg = f"a line rising by 2 per step has slope 2 once 3 values exist, got {s}"
    assert np.allclose(s, [0, 0, 2, 2]), msg


test_slope_of_a_straight_line()


def test_slope_of_two_points():
    assert np.allclose(rolling_slope(np.array([1., 2., 4., 8.]), 2), [0, 1, 2, 4])


test_slope_of_two_points()


def test_ridge_without_penalty_is_least_squares():
    Xc, yc = np.array([[-1.], [0.], [1.]]), np.array([1., 3., 5.])
    w0, m0 = fit_ridge(Xc, yc, 0.0)
    ok = np.allclose(w0, [2]) and np.isclose(m0, 3)
    assert ok, f"expected w = [2], y_mean = 3, got {w0}, {m0}"
    w2, _ = fit_ridge(Xc, yc, 2.0)
    assert np.allclose(w2, [1]), "lambda = 2 halves the weight here: (2 / (2 + 2)) * 2"


test_ridge_without_penalty_is_least_squares()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
X_all = make_features(train, rolling_slope)

# training rows: the capped target; validation points: the true, uncapped RUL
X_tr = X_all[~is_val].to_numpy()
y_tr = train.loc[~is_val, "y"].to_numpy()
X_val = X_all.loc[idx_val].to_numpy()
y_val = train.loc[idx_val, "RUL"].to_numpy()

# the 14 sensors of the current cycle, for the "raw" baseline
raw_tr = train.loc[~is_val, SENSORS].to_numpy(float)
raw_val = train.loc[idx_val, SENSORS].to_numpy(float)
print("training rows", X_tr.shape, "| validation points", X_val.shape,
      "| true RUL range", y_val.min(), y_val.max())

**Questions** (answer them in `practice_7_answers.md`, Task 3, with the numbers your code printed):

1. Report RMSE and score of the three baselines on the validation points.
2. Check `"rolling"` against `sklearn.linear_model.Ridge(alpha=1.0)`: how far apart are the validation predictions?
3. What do the slopes tell a linear model that one cycle cannot? Which change to `make_features` would leak the future, and what would the validation RMSE then tell you?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground: a fitted model is the tuple `(f0, lr, trees)`, and this is how it predicts.*

In [15]:
def boost_predict(model, X):
    f0, lr, trees = model
    return f0 + lr * np.sum([t.predict(X) for t in trees], axis=0)

---

## Task 4. Gradient boosting with an asymmetric loss  
*0.25 points: 0.15 for the code (tests), 0.10 for the answers*

**Gradient boosting** builds $F_M(\mathbf{x}) = f_0 + \nu \sum_{m=1}^{M} h_m(\mathbf{x})$. Each
tree $h_m$ is fitted to the **negative gradient** of the loss at the current prediction: a step
of gradient descent taken in the space of functions, with a tree as the step direction and
$\nu$ as the step size.

We make late predictions dearer. With $d = \hat y - y$ and a weight $r \ge 1$ for the late side,

$$L(d) = \tfrac12\, w(d)\, d^2, \qquad w(d) = \begin{cases} r, & d > 0, \\ 1, & d \le 0, \end{cases}
\qquad\Rightarrow\qquad r_i = -\frac{\partial L}{\partial \hat y_i} = -\,w(d_i)\, d_i.$$

One round: compute $r_i$ on all rows at the current $\hat y$, fit a tree on a random share of the
rows, and update every prediction, $\hat y \leftarrow \hat y + \nu\, h_m(\mathbf{x})$. Start from
$f_0$ = the median of $\mathbf{y}$.

For $r = 1$ this is ordinary boosting with squared error, which converges to the conditional
mean. For $r > 1$ the minimiser of the expected loss solves $r\,E[(\hat y - y)_+] = E[(y - \hat
y)_+]$: the prediction sits where the late mass, weighted by $r$, balances the early mass. This
point is called an **expectile**, the squared-loss cousin of a quantile. On constant input with
targets $\{0, 10\}$ and $r = 2$ the condition reads $2p = 10 - p$, so $p = 10/3$ instead of the
mean 5.

**Write**

- `asym_residual(y, p, ratio)`: the negative gradient $r_i$ for arrays `y` (true) and `p`
  (predicted);
- `boost_fit(X, y, ratio=1.0, n_trees=150, lr=0.1, depth=4, sample=0.5, seed=0)`: returns
  `(f0, lr, trees)` for `boost_predict`. Each tree is a
  `DecisionTreeRegressor(max_depth=depth, min_samples_leaf=20, max_features=0.5)` fitted on a
  random share `sample` of the rows. All randomness comes from `seed`: the same seed gives the
  same model;
- `ratio_table`: a `pd.DataFrame` indexed by `ratio` = 1, 1.5, 2, 3 with columns `"RMSE"`,
  `"score"`, `"mean error"`: the means over seeds 0, 1, 2 of each metric on the validation points,
  for models fitted on `X_tr`, `y_tr`, predictions clipped to $[0, 125]$.

In [ ]:
def asym_residual(y, p, ratio):
    ...


def boost_fit(X, y, ratio=1.0, n_trees=150, lr=0.1, depth=4, sample=0.5, seed=0):
    ...


ratio_table = ...

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_residual_weights_the_late_side():
    r = asym_residual(np.array([10., 10.]), np.array([12., 7.]), ratio=3)
    msg = f"late by 2 with ratio 3 gives -6, early by 3 gives +3; got {r}"
    assert np.allclose(r, [-6, 3]), msg


test_residual_weights_the_late_side()


def test_constant_input_goes_to_the_mean():
    Xc, yc = np.zeros((100, 1)), np.array([0.0] * 50 + [10.0] * 50)
    model = boost_fit(Xc, yc, ratio=1.0, n_trees=200, sample=1.0)
    got = boost_predict(model, Xc)[0]
    msg = f"with ratio 1 the fit on constant input goes to the mean 5, got {got:.3f}"
    assert abs(got - 5) < 0.02, msg


test_constant_input_goes_to_the_mean()


def test_constant_input_goes_to_the_expectile():
    Xc, yc = np.zeros((100, 1)), np.array([0.0] * 50 + [10.0] * 50)
    model = boost_fit(Xc, yc, ratio=2.0, n_trees=200, sample=1.0)
    got = boost_predict(model, Xc)[0]
    msg = f"with ratio 2 it goes to the expectile 10/3, got {got:.3f}"
    assert abs(got - 10 / 3) < 0.02, msg


test_constant_input_goes_to_the_expectile()


def test_ratio_table_shape():
    assert isinstance(ratio_table, pd.DataFrame), "ratio_table: a DataFrame"
    assert list(ratio_table.index) == [1, 1.5, 2, 3], "ratio_table: index 1, 1.5, 2, 3"
    assert {"RMSE", "score", "mean error"} <= set(ratio_table.columns)


test_ratio_table_shape()


print("all tests passed")

**Questions** (answer them in `practice_7_answers.md`, Task 4, with the numbers your code printed):

1. For each ratio report the mean over the three seeds of RMSE, score and mean error $\bar d$, and the spread between seeds.
2. Check the symmetric case: the RMSE of `ratio = 1` against `sklearn.ensemble.HistGradientBoostingRegressor` on the same data.
3. Compare with your ridge from Task 3. What did the trees add?
4. What does a larger ratio do to $\bar d$, to RMSE and to the score? Are the differences larger than the spread between seeds? Which ratio would you choose? (From Task 6 on, the common cell uses 1.5 for everybody.)

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 5. The test set, once  
*0.15 points: 0.06 for the code (tests), 0.09 for the answers*

One RMSE hides where the model is wrong. Split the true RUL into ranges $(\text{lo}, \text{hi}]$ and
report, for each range, the number of engines, the RMSE and the mean error $\bar d$. RMSE squares
the errors, so one range with large errors can dominate the total. The mean error shows the
direction: a range with $\bar d < 0$ is predicted early on average.

Fix your choices before you run anything on the test set.

**Write**

- `error_by_range(y_true, y_pred, edges)`: for `edges = [0, 25, 50]` the ranges $(0, 25]$ and
  $(25, 50]$; returns a list of `(n, rmse, mean error)`, one tuple per range;
- `test_rmse`: a dict with the RMSE against `rul_test` at the **last cycle** of each test engine of
  three models trained on all 100 training engines: `"constant"` (the mean of the capped target),
  `"ridge"` (rolling features, $\lambda = 1$) and `"boosting"` (`boost_fit`, your ratio from Task 4,
  seed 0). Predictions clipped to $[0, 125]$.

In [ ]:
def error_by_range(y_true, y_pred, edges):
    ...


test_rmse = {"constant": ..., "ridge": ..., "boosting": ...}

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_two_ranges():
    got = error_by_range([10, 20, 30, 40], [12, 18, 33, 37], [0, 20, 40])
    rounded = [(n, round(r, 3), round(m, 3)) for n, r, m in got]
    assert rounded == [(2, 2.0, 0.0), (2, 3.0, 0.0)], got


test_two_ranges()


def test_ranges_are_closed_on_the_right():
    got = error_by_range([20, 21], [25, 21], [0, 20, 40])
    assert [n for n, _, _ in got] == [1, 1], "20 belongs to (0, 20], 21 to (20, 40]"
    assert np.isclose(got[0][2], 5.0), "mean error of the first range: 25 - 20"


test_ranges_are_closed_on_the_right()


def test_test_rmse_reported():
    assert set(test_rmse) == {"constant", "ridge", "boosting"}
    assert all(v > 0 for v in test_rmse.values())


test_test_rmse_reported()


print("all tests passed")

**Questions** (answer them in `practice_7_answers.md`, Task 5, with the numbers your code printed):

1. Report RMSE and score against `rul_test` for the three models, and the RMSE of boosting a second time against `np.minimum(rul_test, 125)`, which many papers use.
2. Report the RMSE of boosting for true RUL in 0–25, 25–50, 50–100 and above 100, and plot predicted against true RUL. Which range is predicted best, and is that the range that matters?
3. Print the five worst engines. What do they have in common?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Tasks 6 and 7: the held-out predictions of the boosting model with `ratio = 1.5`, and the alarm window. Run it as it is; it takes about a minute.*

In [ ]:
BEST_RATIO = 1.5           # the same for everybody, so that Tasks 6 and 7 can be compared
L_MIN, L_MAX = 15, 60            # an alarm counts as on time if 15 to 60 cycles remain


def oof_predict(ratio=BEST_RATIO, seed=0):
    # Held-out RUL prediction for every cycle of every training engine, 5 folds by engine.
    X, y = X_all.to_numpy(), train["y"].to_numpy()
    fold = train["fold"].to_numpy()
    out = np.zeros(len(train))
    for f in range(5):
        model = boost_fit(X[fold != f], y[fold != f], ratio=ratio, seed=seed)
        out[fold == f] = boost_predict(model, X[fold == f])
    return pd.Series(np.clip(out, 0, CAP), index=train.index)


oof = oof_predict()
print("out-of-fold predictions:", oof.shape)

---

## Task 6. From a prediction to an alarm: a one-sensor limit against the model  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

A planner needs time to get a spare engine: an alarm with less than 15 cycles left is late. An
alarm with more than 60 cycles left wastes engine life and counts as premature. So the outcome
for one engine is *on time* if $15 \le \text{lead} \le 60$.

A single reading above a limit is noise, so an alarm is **confirmed**: it fires at the first cycle
that completes `confirm` consecutive cycles over the limit. If one noisy cycle crosses the limit
with probability $q$, three in a row do so with probability about $q^3$: at $q = 0.05$ that is
$1.25 \cdot 10^{-4}$. The price is a delay of at least `confirm` − 1 cycles.

At the alarm cycle $i$ of an engine with $n$ rows, the **lead** is the number of cycles left:
$n - 1 - i$.

| | |
|---|---|
| no alarm ever | `missed` |
| lead > `l_max` | `premature` |
| lead < `l_min` | `late` |
| otherwise | `on time` |

**The classical alarm.** For each of the 14 sensors: smooth with a rolling mean of 5 cycles; take
the direction in which it moves towards failure; alarm above $\mu + 3\sigma$ of the first 30 cycles
of the engines in the *other* four folds. Keep the best sensor (picked in hindsight, which is
generous to the alarm).

**The model alarm.** Alarm when `oof` (the held-out predicted RUL) is at or below a threshold $T$.
For each fold, choose $T$ from 10 to 60 on the *other* four folds, the smallest $T$ among the best,
and apply it to this fold.

**Write**

- `first_alarm(flags, confirm=3)`: the index of the first position that completes `confirm`
  consecutive `True` in a boolean array, or `None`;
- `outcome(flags, confirm=3, l_min=15, l_max=60)`: `(label, lead)` with the labels of the table;
  `lead` is `None` when there is no alarm;
- `on_time`: a dict with the number of the 100 training engines whose alarm is on time,
  `"sensor"` for the best sensor and `"model"` for the model alarm.

In [ ]:
def first_alarm(flags, confirm=3):
    ...


def outcome(flags, confirm=3, l_min=15, l_max=60):
    ...


on_time = {"sensor": ..., "model": ...}

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_first_confirmed_alarm():
    f = np.array([0, 1, 1, 0, 1, 1, 1, 1, 0, 0], bool)
    got = first_alarm(f)
    assert got == 6, f"three in a row are completed at index 6, got {got}"
    assert first_alarm(np.zeros(5, bool)) is None


test_first_confirmed_alarm()


def test_outcome_labels():
    f = np.array([0, 1, 1, 0, 1, 1, 1, 1, 0, 0], bool)
    assert outcome(f, l_min=2, l_max=4) == ("on time", 3)
    assert outcome(f, l_min=4, l_max=9)[0] == "late"
    assert outcome(f, l_min=1, l_max=2)[0] == "premature"
    assert outcome(np.zeros(5, bool)) == ("missed", None)


test_outcome_labels()


def test_on_time_counts():
    assert set(on_time) == {"sensor", "model"}
    assert all(0 <= int(v) <= 100 for v in on_time.values()), "a count out of 100 engines"


test_on_time_counts()


print("all tests passed")

**Questions** (answer them in `practice_7_answers.md`, Task 6, with the numbers your code printed):

1. Print a table with the best sensor alarm and the model alarm: on time, late, missed, premature, the mean lead of the alarms on time and of all alarms. Which sensor is best?
2. POD is the share of the 100 engines with an on-time alarm. Report both.
3. The lead of an alarm is engine life you throw away. How many cycles per engine does each rule waste, as a share of the median lifetime? Why might a plant still prefer the rule with the larger waste?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Task 7: the FMEA table. Run it as it is.*

In [26]:
# S and O are assumed for this practice.
# D of the other four modes comes from inspection practice.
FMEA = pd.DataFrame(
    [("HPC degradation",        8, 6, np.nan),
     ("Fan blade erosion",      6, 5, 6),
     ("LPT blade creep",        9, 3, 7),
     ("Main bearing spalling", 10, 2, 6),
     ("Fuel nozzle coking",     5, 4, 5)],
    columns=["failure mode", "S", "O", "D other"]).set_index("failure mode")
print(FMEA.to_string())

                        S  O  D other
failure mode                         
HPC degradation         8  6      NaN
Fan blade erosion       6  5    6.000
LPT blade creep         9  3    7.000
Main bearing spalling  10  2    6.000
Fuel nozzle coking      5  4    5.000


---

## Task 7. FMEA: the risk priority number before and after  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

With $k$ detections out of $n$ engines, the point estimate $\hat p = k/n$ ignores how few engines
you tested. The **Wilson lower bound** at confidence $z$ ($z = 1.96$ for 95 %) is

$$p_{\text{low}} = \frac{\hat p + \dfrac{z^2}{2n} - z\sqrt{\dfrac{\hat p(1-\hat p)}{n} + \dfrac{z^2}{4n^2}}}{1 + \dfrac{z^2}{n}}.$$

Where does it come from? $\hat p$ scatters around the true $p$ with standard deviation
$\sqrt{p(1-p)/n}$. The Wilson interval is the set of all $p$ for which the observed $\hat p$ lies
within $z$ of those standard deviations, $|\hat p - p| \le z\sqrt{p(1-p)/n}$. Squaring gives a
quadratic in $p$, and the formula is its smaller root. Because the spread uses the unknown $p$
rather than $\hat p$, the bound stays sensible at $\hat p = 1$, where the naive interval
$\hat p \pm z\sqrt{\hat p(1-\hat p)/n}$ collapses to the single point 1.

The detection rating of this course, the same scale as in `inclass_work_11`:

| POD at least | 0.99 | 0.95 | 0.90 | 0.80 | 0.70 | 0.50 | 0.30 | 0.15 | 0.05 | below |
|---|---|---|---|---|---|---|---|---|---|---|
| D | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |

$\text{RPN} = S \times O \times D$. $D$ of HPC degradation is yours to measure from the Wilson
lower bound of POD on the 100 engines: **before** with your best one-sensor alarm of Task 6,
**after** with your model alarm.

**Write** `wilson_lower(k, n, z=1.96)`, `detection_rating(p)` (an `int` from 1 to 10),
`rpn(S, O, D)`, and `D_hpc`: a dict `{"before": ..., "after": ...}` with the two detection ratings.

In [ ]:
def wilson_lower(k, n, z=1.96):
    ...


def detection_rating(p):
    ...


def rpn(S, O, D):
    ...


D_hpc = {"before": ..., "after": ...}

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_wilson_known_values():
    for (k, n), want in {(100, 100): 0.9630, (85, 100): 0.7672, (5, 10): 0.2366}.items():
        got = wilson_lower(k, n)
        msg = f"wilson_lower({k}, {n}): expected {want}, got {got:.4f}"
        assert abs(got - want) < 5e-4, msg


test_wilson_known_values()


def test_rating_scale():
    got = [detection_rating(p) for p in (0.995, 0.99, 0.72, 0.0499, 0.01)]
    assert got == [1, 1, 5, 10, 10], f"expected [1, 1, 5, 10, 10], got {got}"
    assert rpn(8, 6, 5) == 240


test_rating_scale()


def test_ratings_reported():
    assert set(D_hpc) == {"before", "after"}
    assert all(1 <= int(v) <= 10 for v in D_hpc.values())


test_ratings_reported()


print("all tests passed")

**Questions** (answer them in `practice_7_answers.md`, Task 7, with the numbers your code printed):

1. Fill the table: POD, its lower bound, D and RPN for HPC degradation before and after, and the RPN of the other four modes. Print the ranking before and after.
2. What would D have been with the point estimate of POD instead of the lower bound? How many engines would you need to claim D = 1?
3. Did the model reduce the risk of an HPC failure happening? What exactly has changed?
4. FD001 contains only one failure mode. May you lower D of "Fan blade erosion" because the model watches the same sensors? What test would you ask for?
5. Compare "Main bearing spalling" (S = 10) with "Fan blade erosion" (S = 6) in your table. Does the RPN order them the way a safety engineer would? What rule would you add?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

### Sources

- Saxena A., Goebel K., Simon D., Eklund N. Damage propagation modeling for aircraft engine
  run-to-failure simulation. Proc. International Conference on Prognostics and Health Management,
  2008. (the simulator, the data, the score)
- Heimes F.O. Recurrent neural networks for remaining useful life estimation. Proc. International
  Conference on Prognostics and Health Management, 2008. (the piecewise-linear target)
- AIAG & VDA. FMEA Handbook, 1st ed., 2019. (Action Priority in place of RPN)